In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Final Test Evaluation on Best Performing Model
## Goal: Evaluate best performing model on untouched test split to acess unseen and real data performance.

In [ ]:
class ModelEvaluator:

    def __init__(
        self,
        champion_model,
        processor,
        optimal_threshold=0.48,
        model_name="Logistic Regression",
    ):
        self.model = champion_model
        self.processor = processor
        self.threshold = optimal_threshold
        self.model_name = model_name

    def evaluate_test_set(self, X_test_raw, y_test_raw):
        # Transform X_test and evaluate best model at optimal threshold
        X_test_proc, self.y_test = self.processor.transform(
            X_test_raw, y_test_raw
        )

        self.probs = self.model.predict_proba(X_test_proc)[:, 1]
        self.preds = (self.probs >= self.threshold).astype(int)

        metrics = {
            "Model": self.model_name,
            "Optimal Threshold": self.threshold,
            "Accuracy": round(accuracy_score(self.y_test, self.preds), 4),
            "Precision": round(
                precision_score(
                    self.y_test, self.preds, zero_division=0
                ),
                4,
            ),
            "Recall": round(
                recall_score(self.y_test, self.preds, zero_division=0),
                4,
            ),
            "F1-Score": round(
                f1_score(self.y_test, self.preds, zero_division=0), 4
            ),
            "ROC-AUC": round(roc_auc_score(self.y_test, self.probs), 4),
        }

        return pd.DataFrame([metrics])

    def plot_test_confusion_matrix(self):
        # Plot COnfusion matrix
        cm = confusion_matrix(self.y_test, self.preds)

        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            ax=ax,
            xticklabels=["No Stroke", "Stroke"],
            yticklabels=["No Stroke", "Stroke"],
        )
        ax.set_title(
            f"Final Test Set: {self.model_name}\nThreshold: {self.threshold:.2f}"
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        plt.tight_layout()
        plt.show()

    def plot_diagnostic_curves(self):
        #Plots ROC-AUC and Precision-Recall curves on test set
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        RocCurveDisplay.from_predictions(
            self.y_test, self.probs, ax=axes[0], name=self.model_name
        )
        axes[0].set_title("ROC Curve (Test Set)")
        axes[0].plot([0, 1], [0, 1], "k--", label="Random Classifier")

        PrecisionRecallDisplay.from_predictions(
            self.y_test, self.probs, ax=axes[1], name=self.model_name
        )
        axes[1].set_title("Precision-Recall Curve (Test Set)")

        plt.tight_layout()
        plt.show()